# LLM + Prompt Tracking with MLflow
Using `google/flan-t5-small` — runs on CPU, fast, ~80MB download.

In [ ]:
!pip install transformers torch sentencepiece mlflow --quiet

In [8]:
import time
import torch
import mlflow
from transformers import T5ForConditionalGeneration, T5Tokenizer

## 1. Setup MLflow

In [9]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("llm-prompt-tracking")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/06/16 11:25:08 INFO mlflow.tracking.fluent: Experiment with name 'llm-prompt-tracking' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


## 2. Load Model (FLAN-T5-small)
Downloads ~80MB on first run, cached afterwards.

In [ ]:
MODEL_NAME = "google/flan-t5-small"

print("Loading model...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.eval()
print("Model ready!")

## 3. Inference Helper

In [ ]:
def generate(prompt, max_new_tokens=100, temperature=1.0):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature != 1.0),
            temperature=temperature if temperature != 1.0 else None,
        )
    latency = time.time() - start
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    input_tokens = inputs["input_ids"].shape[1]
    output_tokens = outputs.shape[1]
    return response, latency, input_tokens, output_tokens

## 4. Experiment 1 — Track Different Prompts
Each prompt runs as a separate MLflow run so you can compare them in the UI.

In [ ]:
prompts = [
    ("translation",   "Translate English to French: The weather is nice today."),
    ("qa",            "Answer the question: What is the capital of France?"),
    ("summarization", "Summarize: Machine learning helps computers learn patterns from data without being explicitly programmed."),
    ("sentiment",     "Classify the sentiment as positive or negative: I really enjoyed this movie, it was fantastic!"),
]

for task, prompt in prompts:
    with mlflow.start_run(run_name=f"prompt_{task}"):
        response, latency, in_tok, out_tok = generate(prompt)

        # Log prompt config
        mlflow.log_param("model", MODEL_NAME)
        mlflow.log_param("task", task)
        mlflow.log_param("prompt", prompt)
        mlflow.log_param("response", response)

        # Log metrics
        mlflow.log_metric("latency_sec", round(latency, 3))
        mlflow.log_metric("input_tokens", in_tok)
        mlflow.log_metric("output_tokens", out_tok)

        print(f"[{task}]")
        print(f"  Prompt  : {prompt}")
        print(f"  Response: {response}")
        print(f"  Latency : {latency:.2f}s | in={in_tok} out={out_tok} tokens\n")

## 5. Experiment 2 — Track Prompt Versions
Compare how rephrasing the same question changes the output.

In [ ]:
prompt_versions = {
    "v1_basic":    "What is machine learning?",
    "v2_detailed": "Explain machine learning in simple terms for a beginner.",
    "v3_analogy":  "Explain machine learning using an analogy that a child can understand.",
}

records = []  # for log_table later

for version, prompt in prompt_versions.items():
    with mlflow.start_run(run_name=f"prompt_{version}"):
        response, latency, in_tok, out_tok = generate(prompt, max_new_tokens=150)

        mlflow.log_param("model", MODEL_NAME)
        mlflow.log_param("prompt_version", version)
        mlflow.log_param("prompt", prompt)
        mlflow.log_param("response", response)
        mlflow.log_metric("latency_sec", round(latency, 3))
        mlflow.log_metric("input_tokens", in_tok)
        mlflow.log_metric("output_tokens", out_tok)

        records.append({
            "version": version,
            "prompt": prompt,
            "response": response,
            "latency_sec": round(latency, 3),
        })

        print(f"[{version}]")
        print(f"  Prompt  : {prompt}")
        print(f"  Response: {response}")
        print(f"  Latency : {latency:.2f}s\n")

## 6. Log a Comparison Table
`mlflow.log_table()` saves a structured table you can view as an artifact in the UI.

In [ ]:
with mlflow.start_run(run_name="prompt_comparison_table"):
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_table(
        data=records,
        artifact_file="prompt_comparison.json"
    )
    print("Comparison table logged!")

import pandas as pd
pd.DataFrame(records)

## 7. Experiment 3 — Sweep max_new_tokens
See how output length affects latency.

In [ ]:
prompt = "Describe the benefits of renewable energy."

for max_tok in [50, 100, 200]:
    with mlflow.start_run(run_name=f"max_tokens_{max_tok}"):
        response, latency, in_tok, out_tok = generate(prompt, max_new_tokens=max_tok)

        mlflow.log_param("model", MODEL_NAME)
        mlflow.log_param("prompt", prompt)
        mlflow.log_param("max_new_tokens", max_tok)
        mlflow.log_param("response", response)
        mlflow.log_metric("latency_sec", round(latency, 3))
        mlflow.log_metric("output_tokens", out_tok)

        print(f"max_new_tokens={max_tok} | latency={latency:.2f}s | out_tokens={out_tok}")
        print(f"  Response: {response}\n")

## Done!
Open the MLflow UI to explore all runs:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Then go to `http://127.0.0.1:5000` → experiment **llm-prompt-tracking**.

In [11]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("llm-prompt-tracking")
client.delete_experiment(exp.experiment_id)  # soft-delete if not already

# Permanently purge from DB
# Run this in terminal:
# mlflow gc --backend-store-uri sqlite:///mlflow.db
